# VM2 NEW — Can Cal-QL t12i / prefill 온라인 6개

전용 G4 VM에서 **0부터 9까지 순서대로** 실행합니다. 1번은 런타임을 재시작하므로 재시작 뒤 2번부터 계속합니다. 이 노트북은 `can_calql_{t12i,prefill}_s{1,2,3}`만 실행합니다.

## 0. Drive 마운트와 GPU 확인

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Conda 설치 — 실행하면 런타임 자동 재시작

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()

## 2. 재시작 후 Drive 재마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJ = '/content/drive/MyDrive/dsrl_project'
print(PROJ)

## 3. 최신 o2o 저장소 준비

In [ ]:
%%bash
set -e
git config --global url."https://github.com/".insteadOf "git@github.com:"
if [ -d /content/dsrl/.git ]; then
  git -C /content/dsrl checkout o2o
  git -C /content/dsrl pull --ff-only origin o2o
else
  test ! -e /content/dsrl || { echo '/content/dsrl exists but is not a git checkout'; exit 2; }
  git clone --recurse-submodules -b o2o https://github.com/msp0617/dsrl.git /content/dsrl
fi
git -C /content/dsrl submodule sync --recursive
git -C /content/dsrl submodule update --init --recursive
echo -n 'HEAD: '; git -C /content/dsrl rev-parse --short HEAD

## 4. 캐시에서 conda 환경 복원

In [ ]:
%%bash
set -e
CACHE=/content/drive/MyDrive/dsrl_project/env_cache/dsrl_env.tar.gz
test -s "$CACHE" || { echo "missing $CACHE"; exit 2; }
mkdir -p /usr/local/envs
rm -rf /usr/local/envs/dsrl
tar -xzf "$CACHE" -C /usr/local/envs
source /usr/local/etc/profile.d/conda.sh
conda activate dsrl
python - <<'PY'
import torch, robomimic, robosuite, mujoco, stable_baselines3
assert torch.cuda.is_available(), 'GPU runtime required'
print('torch', torch.__version__, '| GPU', torch.cuda.get_device_name(0))
PY
python /usr/local/envs/dsrl/lib/python3.10/site-packages/robosuite/scripts/setup_macros.py

## 5. Can 정책·정규화 복원과 환경 패치

In [ ]:
%%bash
set -e
source /usr/local/etc/profile.d/conda.sh && conda activate dsrl
PROJ=/content/drive/MyDrive/dsrl_project
RUNTIME=/content/dsrl/dppo/log
DRIVE=$PROJ/dppo_log
mkdir -p "$RUNTIME"
test -d "$DRIVE" || { echo "missing $DRIVE"; exit 2; }
cp -r "$DRIVE"/. "$RUNTIME"/
CKPT_REL=robomimic-pretrain/can/can_pre_diffusion_mlp_ta4_td20/2024-06-28_13-29-54/checkpoint/state_5000.pt
NORM_REL=robomimic/can/normalization.npz
CKPT_SRC=$(find "$RUNTIME" "$DRIVE" -type f -path "*/$CKPT_REL" -print -quit)
NORM_SRC=$(find "$RUNTIME" "$DRIVE" -type f -path "*/$NORM_REL" -print -quit)
test -n "$CKPT_SRC" -a -n "$NORM_SRC" || { echo 'Can policy assets missing'; exit 2; }
mkdir -p "$RUNTIME/$(dirname "$CKPT_REL")" "$RUNTIME/$(dirname "$NORM_REL")"
[ "$CKPT_SRC" = "$RUNTIME/$CKPT_REL" ] || cp -f "$CKPT_SRC" "$RUNTIME/$CKPT_REL"
[ "$NORM_SRC" = "$RUNTIME/$NORM_REL" ] || cp -f "$NORM_SRC" "$RUNTIME/$NORM_REL"
cat > /content/env.sh <<'EOS'
export MUJOCO_GL=egl
export PYOPENGL_PLATFORM=egl
export WANDB_MODE=disabled
EOS
python /content/dsrl/colab/patch_env.py
ls -lh "$RUNTIME/$CKPT_REL" "$RUNTIME/$NORM_REL"

## 6. Cal-QL 사전학습 3개와 prefill 데이터 사전검사

In [ ]:
%%bash
set -e
source /usr/local/etc/profile.d/conda.sh && conda activate dsrl
python - <<'PY'
import gc, os, numpy as np, torch
proj = '/content/drive/MyDrive/dsrl_project'
for seed in (1, 2, 3):
    path = f'{proj}/logs/pretrain/calql_can_s{seed}.pt'
    assert os.path.getsize(path) > 1_000_000, f'missing or short: {path}'
    payload = torch.load(path, map_location='cpu', weights_only=False)
    got = (payload.get('meta') or {}).get('method')
    assert got == 'calql', f'{path}: method={got}, expected calql'
    print('OK', os.path.basename(path), 'method', got)
    del payload; gc.collect()
path = f'{proj}/offline/can_train_offline.npz'
with np.load(path) as data:
    assert len(data['states']) > 0
    print('OK', os.path.basename(path), 'rows', len(data['states']))
PY

## 7. t12i 3개 + prefill 3개 시작

In [ ]:
%%bash
set -e
source /usr/local/etc/profile.d/conda.sh && conda activate dsrl
source /content/env.sh
cd /content/dsrl
git pull --ff-only origin o2o
PROJ=/content/drive/MyDrive/dsrl_project
mkdir -p "$PROJ/logs"
CFG='--config-path=cfg/robomimic --config-name=dsrl_can.yaml'
launch () {
  EXP=$1; shift
  if pgrep -af '[t]rain_dsrl.py' | grep -Fq "exp_id=$EXP"; then echo "already running: $EXP"; return; fi
  nohup python train_dsrl.py $CFG exp_id=$EXP "$@" > "$PROJ/logs/$EXP.out" 2>&1 &
  echo "started $EXP (pid $!)"
}
for S in 1 2 3; do
  PT=$PROJ/logs/pretrain/calql_can_s$S.pt
  launch can_calql_t12i_s$S seed=$S variant=calql pretrain_path=$PT log_dir=$PROJ/logs train.total_env_steps=150000 offline_mix.mode=none load_offline_data=False train.ent_coef=auto_0.3 train.target_ent=12
  launch can_calql_prefill_s$S seed=$S variant=calql pretrain_path=$PT log_dir=$PROJ/logs train.total_env_steps=150000 offline_mix.mode=prefill offline_data_path=$PROJ/offline/can_train_offline.npz
done

## 8. 3분 후 자동 확인 — 6개가 running이고 ERR가 없어야 함

In [ ]:
%%bash
sleep 180
source /usr/local/etc/profile.d/conda.sh && conda activate dsrl
cd /content/dsrl
PROJ=/content/drive/MyDrive/dsrl_project
python scripts/inspect_runs.py --proj "$PROJ" --only can_calql_t12i_s,can_calql_prefill_s
for KIND in t12i prefill; do for S in 1 2 3; do
  E=can_calql_${KIND}_s$S; echo "== $E"
  grep '\[pretrain\]\|\[eval\] env_steps=0\|Traceback\|Error' "$PROJ/logs/$E.out" | tail -n 4 || true
done; done
echo '== processes'; pgrep -af '[t]rain_dsrl.py' || true

## 9. Keepalive — 마지막 프로세스가 끝나면 자동 반납

8번에서 오류가 없을 때만 실행하고 이 셀을 계속 실행 상태로 둡니다.

In [ ]:
import subprocess, time
from pathlib import Path
EXPECTED = [f'can_calql_{kind}_s{s}' for kind in ('t12i', 'prefill') for s in (1, 2, 3)]
LOGS = Path('/content/drive/MyDrive/dsrl_project/logs')

def running():
    out = subprocess.run(['ps', '-eo', 'pid,args'], capture_output=True, text=True).stdout
    return [line.strip() for line in out.splitlines() if 'train_dsrl.py' in line and any(f'exp_id={e}' in line for e in EXPECTED)]

def last_event(exp):
    path = LOGS / f'{exp}.out'
    if not path.exists(): return 'NO .out'
    lines = path.read_text(errors='replace').splitlines()[-500:]
    for line in reversed(lines):
        if any(x in line for x in ('[eval]', '[done]', 'Traceback', 'Error')): return line[:100]
    return 'starting'

while True:
    procs = running()
    print(time.strftime('%H:%M'), f'running {len(procs)}/{len(EXPECTED)}', '|', ' | '.join(f'{e}: {last_event(e)}' for e in EXPECTED), flush=True)
    if not procs:
        print('all VM2 runs stopped -> unassigning', flush=True)
        from google.colab import runtime
        runtime.unassign()
        break
    time.sleep(600)